# CartPole - paper's literal replication arms (Colab)

Everything registered in `core/configs.py` under the `paper_*` names, cross-referenced
against `data/paper_ppo_summary.csv` via `PAPER_ARM_TO_RESULTS` - and never actually
trained under DQN by any script in this repo. Two blocks:

- **Block 1 (Output Reuse)** - `paper_hsiao_or_r{4,8,16,32}` (hybrid, Hsiao circuit, 4
  qubits, `n_layers_q=1`), `paper_classical_or_r{4,8,16,32}` (classical control), and the
  entangled variant `paper_hsiao_or_r{4,16}_ent`.
- **Block 3 (ansatz / entanglement)** - `paper_salinas_{1q,2q}_L{1,2,5}`,
  `paper_skolik_{4q,8q}_L{1,2,5}`, and the noent ablation `paper_skolik_4q_L{2,5}_noent`
  (L1 excluded on purpose - FIX-07: on this template the final CZ ring cannot affect a
  PauliZ readout, so at L=1 the entangled and unentangled circuits ARE the same circuit).

**Not included here**: `su2_cartpole_L5` and `cartpole_fourier_ceiling_L5` (exp06's own
pending arms) - those are already queued in the local machine's pipeline (`exp06`,
writing into `exp01`'s directory to reuse `hybrid_fig4`). Training them here too, into a
different directory, would just be duplicate compute answering the same question twice.

## Ordering

Sorted **cheapest to most expensive**, so a Colab session that disconnects partway
still banks the early tiers for certain. The dominant cost driver for a PQC simulated on
CPU is qubit count (the state vector is `2^n_qubits`); circuit depth `L` is a distant
second. So: classical first, then 1q, 2q, 4q (shallow Hsiao, then deeper Skolik), 8q last.

| tier | qubits | arms | why here |
|---|---|---|---|
| 0 | 0 (classical) | `paper_classical_or_r{4,8,16,32}` | no PQC at all |
| 1 | 1 | `paper_salinas_1q_L{1,2,5}` | cheapest possible state vector |
| 2 | 2 | `paper_salinas_2q_L{1,2,5}` | still tiny |
| 3 | 4 | `paper_hsiao_or_r{4,8,16,32}`, `_r{4,16}_ent` | 4 qubits but `n_layers_q=1` fixed - shallow |
| 4 | 4 | `paper_skolik_4q_L{1,2,5}` (+`_noent` at L2/L5) | same qubit count, deeper circuit |
| 5 | 8 | `paper_skolik_8q_L{1,2,5}` | the expensive tier - same qubit count as `hybrid_fig4`, which took the longest of anything already run in this project |

Coverage pass only (**n=3 seeds**, not a conclusion - the standing rule in this repo).
Every cell is checkpointed by its own manifest, so re-running this notebook in a later
session costs nothing for whatever already finished.

---
## 1. Environment

In [ ]:
import os, sys, subprocess, pathlib

GITHUB_USER, REPO_NAME, BRANCH = "RogerMas99", "qrl-dissection", "main"
try:
    from google.colab import userdata
    _tok = userdata.get("GH_TOKEN")
    REPO_URL = (f"https://{_tok}@github.com/{GITHUB_USER}/{REPO_NAME}.git" if _tok
                else f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git")
except Exception:
    REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    CODE    = pathlib.Path("/content/qrl-dissection")
    RESULTS = pathlib.Path("/content/drive/MyDrive/tfm_qrl/results")
else:
    CODE    = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
    RESULTS = pathlib.Path.cwd() / "results"
RES = RESULTS  # alias, see notebooks/README.md

if IN_COLAB:
    if not CODE.exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(CODE)], check=True)
    else:
        subprocess.run(["git", "-C", str(CODE), "pull"], check=True)
    # SimplyQRL is vendored in this repo, so `pip install -e .` is the whole
    # install - see docs/CORRECTIONS.md#fix-04.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(CODE)], check=True)
    # Colab preinstalls jax, and PennyLane imports it opportunistically if it
    # is present - it is absent from both the upstream lock and this repo's
    # requirements.txt (see that file's own note). Uninstall it.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "jax", "jaxlib"], check=False)

sys.path.insert(0, str(CODE / "src"))
assert CODE.exists() and RESULTS is not None

# If jax (or the broken autoray shim FIX-04 documents) is already imported in
# THIS kernel - e.g. Colab's own startup, or a previous run of this cell -
# uninstalling the package does not unload it from memory. Only a runtime
# restart does.
_need_restart = "jax" in sys.modules
if "autoray.autoray" in sys.modules:
    import autoray.autoray as _aa
    _need_restart = _need_restart or not hasattr(_aa, "NumpyMimic")
if _need_restart:
    print("Incompatible modules already loaded -> restarting runtime.")
    os.kill(os.getpid(), 9)

OUTDIR = RESULTS / "exp07_dqn_cartpole_paper_replication"
OUTDIR.mkdir(parents=True, exist_ok=True)
print("CODE   =", CODE)
print("OUTDIR =", OUTDIR)

---
## 2. Preflight

Same gate as `10_dqn_suite_runner.ipynb` - the full test suite, before spending any
compute. Skip only if you already ran it this session.

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"],
                   cwd=str(CODE), capture_output=True, text=True)
print(r.stdout[-2500:])
assert r.returncode == 0, "test suite failing - do not run the grid against it"
print("\npreflight OK")

---
## 3. Shared setup

`DQN_KWARGS` and `steps=100_000` match exp02/exp03 (the other CartPole block sweeps),
not exp01's 60k default - these arms answer the same kind of question those did.
`fix_autoreset=True` only, same precedent: exp01 already measured the FIX-01 contrast
itself, the block sweeps don't repeat it.

In [ ]:
import qrl_dissection
from qrl_dissection.dqn import GreedyEvalConfig, RunSpec, run_grid

print(qrl_dissection.upstream_report())

DQN_KWARGS = {"batch_size": 128, "buffer_size": 10_000, "train_frequency": 10}
STEPS = 100_000
SEEDS = [1, 2, 3]          # coverage pass - not a conclusion
EVAL_EVERY = 10_000
ENV_ID = "CartPole-v1"
EVAL_CFG = GreedyEvalConfig(env_id=ENV_ID, every_steps=EVAL_EVERY, n_episodes=20)

def run_tier(name, arms):
    specs = [RunSpec(arm=a, seed=s, fix_autoreset=True, total_timesteps=STEPS,
                     dqn_kwargs=DQN_KWARGS)
             for a in arms for s in SEEDS]
    print(f"\n=== {name}: {len(arms)} arms x {len(SEEDS)} seeds = {len(specs)} cells ===")
    return run_grid(specs, OUTDIR, env_id=ENV_ID, eval_cfg=EVAL_CFG, claim=False)

### Tier 0 - classical (no PQC). Should finish in minutes.

In [ ]:
TIER0 = [f"paper_classical_or_r{R}" for R in (4, 8, 16, 32)]
_ = run_tier("Tier 0 - classical OR control", TIER0)

### Tier 1 - 1 qubit (`paper_salinas_1q_L{1,2,5}`). Cheapest PQC there is.

In [ ]:
TIER1 = [f"paper_salinas_1q_L{L}" for L in (1, 2, 5)]
_ = run_tier("Tier 1 - Salinas 1q", TIER1)

### Tier 2 - 2 qubits (`paper_salinas_2q_L{1,2,5}`). Still small.

In [ ]:
TIER2 = [f"paper_salinas_2q_L{L}" for L in (1, 2, 5)]
_ = run_tier("Tier 2 - Salinas 2q", TIER2)

### Tier 3 - 4 qubits, Hsiao circuit (`n_layers_q=1` fixed - shallow despite the qubit
count). `paper_hsiao_or_r{4,8,16,32}` plus the entangled variant at R=4,16.

In [ ]:
TIER3 = ([f"paper_hsiao_or_r{R}" for R in (4, 8, 16, 32)]
         + [f"paper_hsiao_or_r{R}_ent" for R in (4, 16)])
_ = run_tier("Tier 3 - Hsiao OR (4q, shallow)", TIER3)

### Tier 4 - 4 qubits, Skolik circuit, depth sweep (`paper_skolik_4q_L{1,2,5}` +
`_noent` ablation at L=2,5 - L=1 excluded, see the FIX-07 note above).

In [ ]:
TIER4 = (["paper_skolik_4q_L1"]
         + ["paper_skolik_4q_L2", "paper_skolik_4q_L2_noent"]
         + ["paper_skolik_4q_L5", "paper_skolik_4q_L5_noent"])
_ = run_tier("Tier 4 - Skolik 4q depth sweep", TIER4)

### Tier 5 - 8 qubits (`paper_skolik_8q_L{1,2,5}`). The expensive one - same qubit
count as `hybrid_fig4`, the slowest arm already run in this project. If the session is
going to disconnect, this is where it happens - and everything above will already be
banked on Drive.

In [ ]:
TIER5 = [f"paper_skolik_8q_L{L}" for L in (1, 2, 5)]
_ = run_tier("Tier 5 - Skolik 8q depth sweep", TIER5)

---
## 4. Resuming / checking progress

Re-running the cells above costs nothing for whatever already has a manifest - the
reuse guard skips completed cells instantly. Run this any time to see the current
state without spending compute:

In [ ]:
import json

by_arm = {}
for mp in sorted(OUTDIR.glob("*.manifest.json")):
    m = json.loads(mp.read_text())
    arm = m.get("spec", {}).get("arm", mp.stem.split("__")[0])
    by_arm.setdefault(arm, set()).add(m.get("spec", {}).get("seed"))

for arm in sorted(by_arm):
    seeds = sorted(s for s in by_arm[arm] if s is not None)
    print(f"{arm:35s} n={len(seeds):2d}  seeds={seeds}")